<a href="https://colab.research.google.com/github/heetaamin/ml-assignment2/blob/main/earlier_version/KNN_modelling_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# kNN Modelling Notebook — Cell 1: Load preprocessed data
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic_knn_preprocessed.csv'
df_knn = pd.read_csv(DATA_PATH)

print("Shape:", df_knn.shape)
print("\nColumns:")
print(df_knn.columns.tolist())
print("\nattack_cat distribution:")
print(df_knn['attack_cat'].value_counts().sort_index())

Mounted at /content/drive
Shape: (160983, 65)

Columns:
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'service_missing', 'proto_arp', 'proto_ospf', 'proto_other', 'proto_tcp', 'proto_udp', 'proto_unas', 'state_ACC', 'state_CLO', 'state_CON', 'state_ECO', 'state_FIN', 'state_INT', 'state_PAR', 'state_REQ', 'state_RST', 'state_URN', 'service_dhcp', 'service_dns', 'service_ftp', 'service_ftp-data', 'service_http', 'service_irc', 'service_pop3', 'service_radius', 'service_smtp', 'service_snmp', 'service_ssh', 'service_ssl']

attack_cat distribution:
attack_cat
0    85487
1     9745
2     1678
3     5236
4    271

In [ ]:
# ============================================================
# Cell 2: Split features/target, set up stratified 5-fold CV
# ============================================================
# X = everything the model can use to predict; y = the actual
# label we're trying to predict. Stratified CV keeps the class
# balance roughly the same in every fold, which matters given
# how imbalanced attack_cat is (Topic 2, slide 129).
#
# 5 folds chosen because the smallest class (category 7, 171
# rows) would only have ~17 rows per fold at 10-fold — too thin
# to get a stable read. 5-fold gives ~34 rows per fold instead.

from sklearn.model_selection import StratifiedKFold

X = df_knn.drop(columns=['attack_cat'])
y = df_knn['attack_cat']

print("X shape:", X.shape)
print("y shape:", y.shape)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Quick sanity check: confirm each fold roughly preserves class balance
for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_num+1} test set class distribution:")
    print(y.iloc[test_idx].value_counts().sort_index())
    break  # just check the first fold for now

X shape: (160983, 64)
y shape: (160983,)

Fold 1 test set class distribution:
attack_cat
0    17098
1     1949
2      335
3     1048
4     5434
5      354
6     4139
7       34
8      292
9     1514
Name: count, dtype: int64


In [ ]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score

In [ ]:
# ============================================================
# Cell 3: Baseline kNN across all 5 folds
# ============================================================
# A reasonable starting point before any tuning — default k,
# default (Euclidean) distance, no class-imbalance handling yet.
# This gives a baseline macro-F1 to compare every later change
# against. random_state=42 kept consistent with the CV split
# above so results are reproducible.

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score

k_baseline = 5

fold_f1_scores = []
fold_balanced_acc = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = KNeighborsClassifier(n_neighbors=k_baseline)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    fold_macro_f1 = f1_score(y_test, y_pred, average='macro')
    fold_bal_acc = balanced_accuracy_score(y_test, y_pred)

    fold_f1_scores.append(fold_macro_f1)
    fold_balanced_acc.append(fold_bal_acc)

    print(f"Fold {fold_num+1}: macro-F1 = {fold_macro_f1:.4f}, balanced accuracy = {fold_bal_acc:.4f}")

print(f"\nMean macro-F1 across 5 folds: {np.mean(fold_f1_scores):.4f} (± {np.std(fold_f1_scores):.4f})")
print(f"Mean balanced accuracy across 5 folds: {np.mean(fold_balanced_acc):.4f} (± {np.std(fold_balanced_acc):.4f})")

Fold 1: macro-F1 = 0.4265, balanced accuracy = 0.4050
Fold 2: macro-F1 = 0.4191, balanced accuracy = 0.3971
Fold 3: macro-F1 = 0.4115, balanced accuracy = 0.3948
Fold 4: macro-F1 = 0.4086, balanced accuracy = 0.3912
Fold 5: macro-F1 = 0.4241, balanced accuracy = 0.4022

Mean macro-F1 across 5 folds: 0.4180 (± 0.0070)
Mean balanced accuracy across 5 folds: 0.3981 (± 0.0050)


In [ ]:
# ============================================================
# Cell 4: Per-class F1 breakdown (baseline, fold 1)
# ============================================================
# The averaged macro-F1 hides WHICH classes are struggling.
# Re-running fold 1 alone here just to inspect this — the full
# baseline numbers above already used all 5 folds properly.

from sklearn.metrics import classification_report

train_idx, test_idx = next(skf.split(X, y))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.834     0.906     0.868     17098
           1      0.422     0.553     0.479      1949
           2      0.033     0.027     0.030       335
           3      0.223     0.129     0.163      1048
           4      0.744     0.737     0.741      5434
           5      0.023     0.008     0.012       354
           6      0.468     0.365     0.410      4139
           7      0.500     0.206     0.292        34
           8      0.506     0.308     0.383       292
           9      0.978     0.811     0.887      1514

    accuracy                          0.731     32197
   macro avg      0.473     0.405     0.426     32197
weighted avg      0.713     0.731     0.719     32197



The strong performers are class 0 and class 9 - the two biggest classes. Class 4 is also reasonable given it is mid sized.

class 2 and 5 are barely functioning, their recall scroes means the model is identifying roughly 1 in 37 of class 2's actual cases, and 1 in 125 of class 5's. It's not weak, it's essentially not detecting these classes at all — almost every real instance of class 2 or class 5 is being misclassified as something else, most likely swallowed into the big neighboring classes.

This happens because with plain majority voting, a query point near the boundary between a huge class and a tiny class will have more class neighbours of the majority class because they are more of them even if it is closer to the minority class.

The gap between accuracy and macro-F1 - 0.731 vs 0.426. This is the imbalance, the overall accuracy looks decent because fo the two huge classes but macro-F1 weighs every class equallu regardless of the size.

Shepard's weighting helps with this problem, by weighing each neighbours bote by inverse distance reduced the raw "more neighbors win" effect, giving genuinely close rare points more relative to farther majority-class points.



In [ ]:
# ============================================================
# Cell 5: Distance-weighted kNN, same 5-fold CV, for comparison
# ============================================================
# Shepard's method (Topic 3, slide 21): weight each neighbor's
# vote by inverse distance, so closer neighbors count more than
# farther ones — instead of every neighbor getting an equal
# vote regardless of distance. Same k=5, same folds as baseline,
# so this is a fair, direct comparison.

fold_f1_weighted = []
fold_balanced_acc_weighted = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = KNeighborsClassifier(n_neighbors=5, weights='distance')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    fold_macro_f1 = f1_score(y_test, y_pred, average='macro')
    fold_bal_acc = balanced_accuracy_score(y_test, y_pred)

    fold_f1_weighted.append(fold_macro_f1)
    fold_balanced_acc_weighted.append(fold_bal_acc)

    print(f"Fold {fold_num+1}: macro-F1 = {fold_macro_f1:.4f}, balanced accuracy = {fold_bal_acc:.4f}")

print(f"\nMean macro-F1 (distance-weighted): {np.mean(fold_f1_weighted):.4f} (± {np.std(fold_f1_weighted):.4f})")
print(f"Mean balanced accuracy (distance-weighted): {np.mean(fold_balanced_acc_weighted):.4f} (± {np.std(fold_balanced_acc_weighted):.4f})")
print(f"\nChange vs. baseline macro-F1: {np.mean(fold_f1_weighted) - np.mean(fold_f1_scores):+.4f}")

Fold 1: macro-F1 = 0.4285, balanced accuracy = 0.4126
Fold 2: macro-F1 = 0.4284, balanced accuracy = 0.4097
Fold 3: macro-F1 = 0.4097, balanced accuracy = 0.3980
Fold 4: macro-F1 = 0.4233, balanced accuracy = 0.4060
Fold 5: macro-F1 = 0.4269, balanced accuracy = 0.4107

Mean macro-F1 (distance-weighted): 0.4234 (± 0.0071)
Mean balanced accuracy (distance-weighted): 0.4074 (± 0.0052)

Change vs. baseline macro-F1: +0.0054


In [ ]:
# ============================================================
# Cell 6: Per-class comparison — did weighting actually help
# the struggling classes specifically?
# ============================================================
train_idx, test_idx = next(skf.split(X, y))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model_weighted = KNeighborsClassifier(n_neighbors=5, weights='distance')
model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)

print(classification_report(y_test, y_pred_weighted, digits=3))

              precision    recall  f1-score   support

           0      0.851     0.893     0.871     17098
           1      0.387     0.535     0.449      1949
           2      0.009     0.012     0.010       335
           3      0.240     0.118     0.159      1048
           4      0.760     0.754     0.757      5434
           5      0.129     0.025     0.042       354
           6      0.468     0.408     0.436      4139
           7      0.400     0.235     0.296        34
           8      0.455     0.329     0.382       292
           9      0.960     0.816     0.882      1514

    accuracy                          0.732     32197
   macro avg      0.466     0.413     0.429     32197
weighted avg      0.723     0.732     0.725     32197



In [ ]:
# ============================================================
# Cell 7: SMOTE — test on one fold first before committing to
# all 5 (computational cost check)
# ============================================================
# SMOTE generates synthetic minority-class rows by interpolating
# between real ones. Applied only inside the training fold, never
# on the test fold, to avoid the same leakage problem as duplicates.
# Moderate target (5000) chosen rather than fully matching the
# majority class (85,487), to keep kNN's runtime reasonable —
# kNN's prediction cost scales with training set size.

from imblearn.over_sampling import SMOTE

train_idx, test_idx = next(skf.split(X, y))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Before SMOTE:")
print(y_train.value_counts().sort_index())

# Only oversample classes currently below 5000, target them to 5000
target_counts = {cls: max(count, 5000) for cls, count in y_train.value_counts().items()}

smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(y_train_smote.value_counts().sort_index())

model_smote = KNeighborsClassifier(n_neighbors=5, weights='distance')
model_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = model_smote.predict(X_test)

print("\n", classification_report(y_test, y_pred_smote, digits=3))

Before SMOTE:
attack_cat
0    68389
1     7796
2     1343
3     4188
4    21736
5     1416
6    16559
7      137
8     1164
9     6058
Name: count, dtype: int64

After SMOTE:
attack_cat
0    68389
1     7796
2     5000
3     5000
4    21736
5     5000
6    16559
7     5000
8     5000
9     6058
Name: count, dtype: int64

               precision    recall  f1-score   support

           0      0.859     0.883     0.870     17098
           1      0.389     0.507     0.440      1949
           2      0.013     0.024     0.017       335
           3      0.236     0.139     0.175      1048
           4      0.783     0.716     0.748      5434
           5      0.091     0.071     0.079       354
           6      0.474     0.392     0.429      4139
           7      0.047     0.353     0.082        34
           8      0.236     0.514     0.323       292
           9      0.966     0.814     0.884      1514

    accuracy                          0.720     32197
   macro avg      0.409   

In [ ]:
# ============================================================
# Cell 8: k + distance metric sweep (single-fold screening)
# ============================================================
# Testing a range of k values against 2 distance metrics
# (Euclidean vs. Manhattan — Topic 3 slide 11), keeping
# weights='distance' fixed since it already showed a real,
# if modest, improvement. Screening on one fold first since a
# full 16-combination x 5-fold sweep would be very slow —
# once we find the best combo here, we confirm it properly
# with all 5 folds.

train_idx, test_idx = next(skf.split(X, y))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

k_values = [1, 3, 5, 7, 9, 15, 21, 31]
metrics = ['euclidean', 'manhattan']

results = []

for metric in metrics:
    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=metric)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        results.append({'k': k, 'metric': metric, 'macro_f1': macro_f1})
        print(f"metric={metric:10s} k={k:3d}  macro-F1={macro_f1:.4f}")

results_df = pd.DataFrame(results)
best = results_df.loc[results_df['macro_f1'].idxmax()]
print(f"\nBest combination: metric={best['metric']}, k={best['k']}, macro-F1={best['macro_f1']:.4f}")

metric=euclidean  k=  1  macro-F1=0.4057
metric=euclidean  k=  3  macro-F1=0.4198
metric=euclidean  k=  5  macro-F1=0.4285
metric=euclidean  k=  7  macro-F1=0.4377
metric=euclidean  k=  9  macro-F1=0.4347
metric=euclidean  k= 15  macro-F1=0.4231
metric=euclidean  k= 21  macro-F1=0.4216
metric=euclidean  k= 31  macro-F1=0.4104
metric=manhattan  k=  1  macro-F1=0.4245
metric=manhattan  k=  3  macro-F1=0.4427
metric=manhattan  k=  5  macro-F1=0.4594
metric=manhattan  k=  7  macro-F1=0.4620
metric=manhattan  k=  9  macro-F1=0.4560
metric=manhattan  k= 15  macro-F1=0.4505
metric=manhattan  k= 21  macro-F1=0.4308
metric=manhattan  k= 31  macro-F1=0.4127

Best combination: metric=manhattan, k=7, macro-F1=0.4620


In [ ]:
# ============================================================
# Cell 9: Confirm winning combination (Manhattan, k=7) with
# proper 5-fold CV
# ============================================================
fold_f1_best = []
fold_balanced_acc_best = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    fold_macro_f1 = f1_score(y_test, y_pred, average='macro')
    fold_bal_acc = balanced_accuracy_score(y_test, y_pred)

    fold_f1_best.append(fold_macro_f1)
    fold_balanced_acc_best.append(fold_bal_acc)

    print(f"Fold {fold_num+1}: macro-F1 = {fold_macro_f1:.4f}, balanced accuracy = {fold_bal_acc:.4f}")

print(f"\nMean macro-F1 (Manhattan, k=7, weighted): {np.mean(fold_f1_best):.4f} (± {np.std(fold_f1_best):.4f})")
print(f"Mean balanced accuracy: {np.mean(fold_balanced_acc_best):.4f} (± {np.std(fold_balanced_acc_best):.4f})")

Fold 1: macro-F1 = 0.4620, balanced accuracy = 0.4428
Fold 2: macro-F1 = 0.4396, balanced accuracy = 0.4250
Fold 3: macro-F1 = 0.4345, balanced accuracy = 0.4229
Fold 4: macro-F1 = 0.4310, balanced accuracy = 0.4174
Fold 5: macro-F1 = 0.4428, balanced accuracy = 0.4262

Mean macro-F1 (Manhattan, k=7, weighted): 0.4420 (± 0.0108)
Mean balanced accuracy: 0.4269 (± 0.0085)



# Progress log — best configuration so far

 Baseline (k=5, euclidean, uniform):        macro-F1 = 0.418
 + distance weighting:                       macro-F1 = 0.423
 + SMOTE (flat target=5000):                 macro-F1 = 0.405 (worse, abandoned)
+ k/metric sweep -> Manhattan, k=7, weighted: macro-F1 = 0.442  <- current best

In [ ]:
# ============================================================
# Cell 10: Wrapper-method test — connection-count cluster
# ============================================================
# Test: does the model perform better, worse, or about the
# same with all 7 cluster features vs. a reduced subset?
# Trying "keep 2" as the reduced option: one source-side count
# (ct_srv_src) and one destination-side count (ct_srv_dst) —
# a simple, defensible representative pair rather than an
# arbitrary cut.

cluster_cols = ['ct_srv_src', 'ct_srv_dst', 'ct_dst_src_ltm', 'ct_src_dport_ltm',
                 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm']
reduced_cluster_cols = ['ct_srv_src', 'ct_srv_dst']  # keep these 2, drop the other 5

drop_cols = [c for c in cluster_cols if c not in reduced_cluster_cols]
X_reduced = X.drop(columns=drop_cols)

print(f"Full feature set: {X.shape[1]} columns")
print(f"Reduced feature set: {X_reduced.shape[1]} columns")

fold_f1_reduced = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_reduced, y)):
    X_train, X_test = X_reduced.iloc[train_idx], X_reduced.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    fold_macro_f1 = f1_score(y_test, y_pred, average='macro')
    fold_f1_reduced.append(fold_macro_f1)
    print(f"Fold {fold_num+1}: macro-F1 = {fold_macro_f1:.4f}")

print(f"\nMean macro-F1 (reduced cluster): {np.mean(fold_f1_reduced):.4f} (± {np.std(fold_f1_reduced):.4f})")
print(f"Change vs. full cluster (0.4420): {np.mean(fold_f1_reduced) - 0.4420:+.4f}")

Full feature set: 64 columns
Reduced feature set: 59 columns
Fold 1: macro-F1 = 0.4637
Fold 2: macro-F1 = 0.4417
Fold 3: macro-F1 = 0.4400
Fold 4: macro-F1 = 0.4407
Fold 5: macro-F1 = 0.4459

Mean macro-F1 (reduced cluster): 0.4464 (± 0.0089)
Change vs. full cluster (0.4420): +0.0044



Progress log — updated

Baseline (k=5, euclidean, uniform):          macro-F1 = 0.418
 + distance weighting:                         macro-F1 = 0.423
 + SMOTE (flat target=5000):                   macro-F1 = 0.405 (worse, abandoned)
 k/metric sweep -> Manhattan, k=7, weighted:   macro-F1 = 0.442
 + connection-cluster reduced (7->2 features): macro-F1 = 0.446  <- current best


# DECISION:
connection-count cluster resolved via wrapper method
 (Topic 2 slide 16) — keep ct_srv_src, ct_srv_dst; drop
 ct_dst_src_ltm, ct_src_dport_ltm, ct_dst_ltm, ct_src_ltm,
 ct_dst_sport_ltm. Empirically confirmed improvement (+0.0044
 macro-F1, tighter fold variance) over keeping all 7.

In [ ]:
# ============================================================
# Cell 11: trans_depth — clamped vs. unclamped
# ============================================================
X_current = X_reduced.copy()  # build on the reduced-cluster set, our new best

X_clamped = X_current.copy()
X_clamped['trans_depth'] = X_clamped['trans_depth'].clip(upper=9)

# Re-scale trans_depth to [0,1] after clamping, consistent with
# how every other feature was already normalized
X_clamped['trans_depth'] = X_clamped['trans_depth'] / X_clamped['trans_depth'].max()

fold_f1_clamped = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_clamped, y)):
    X_train, X_test = X_clamped.iloc[train_idx], X_clamped.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    fold_macro_f1 = f1_score(y_test, y_pred, average='macro')
    fold_f1_clamped.append(fold_macro_f1)
    print(f"Fold {fold_num+1}: macro-F1 = {fold_macro_f1:.4f}")

print(f"\nMean macro-F1 (trans_depth clamped): {np.mean(fold_f1_clamped):.4f} (± {np.std(fold_f1_clamped):.4f})")
print(f"Change vs. unclamped (0.4464): {np.mean(fold_f1_clamped) - 0.4464:+.4f}")

Fold 1: macro-F1 = 0.4635
Fold 2: macro-F1 = 0.4460
Fold 3: macro-F1 = 0.4400
Fold 4: macro-F1 = 0.4405
Fold 5: macro-F1 = 0.4459

Mean macro-F1 (trans_depth clamped): 0.4472 (± 0.0086)
Change vs. unclamped (0.4464): +0.0008



# Progress log — updated, final

 Baseline (k=5, euclidean, uniform):            macro-F1 = 0.418
 + distance weighting:                           macro-F1 = 0.423
 + SMOTE (flat target=5000):                     macro-F1 = 0.405 (worse, abandoned)
 k/metric sweep -> Manhattan, k=7, weighted:     macro-F1 = 0.442
 + connection-cluster reduced (7->2 features):   macro-F1 = 0.446
 + trans_depth clamped/rescaled:                 macro-F1 = 0.447 (negligible, within noise)

# DECISION:
trans_depth left unclamped and unscaled. Tested
 empirically per Lauren's guidance rather than assumed —
 improvement (+0.0008) is well within normal fold variance
 (±0.0086), i.e. no real effect. Its extreme values only touch
 0.049% of rows, so the distortion clamping would prevent was
 always narrowly localized, unlike the connection-count cluster.
 Consistent with the assignment's "avoid unnecessary
 transformations" instruction — no performance case to add one.

In [ ]:
# ============================================================
# Cell 12: Tomek links, standalone (no SMOTE)
# ============================================================
from imblearn.under_sampling import TomekLinks
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

cluster_cols = ['ct_srv_src', 'ct_srv_dst', 'ct_dst_src_ltm', 'ct_src_dport_ltm',
                 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm']
reduced_cluster_cols = ['ct_srv_src', 'ct_srv_dst']
drop_cols = [c for c in cluster_cols if c not in reduced_cluster_cols]

X_reduced = X.drop(columns=drop_cols)
print("X_reduced shape:", X_reduced.shape)

train_idx, test_idx = next(skf.split(X_reduced, y))
X_train, X_test = X_reduced.iloc[train_idx], X_reduced.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Before Tomek:", len(X_train))

tomek = TomekLinks()
X_train_tomek, y_train_tomek = tomek.fit_resample(X_train, y_train)

print("After Tomek:", len(X_train_tomek))
print("\nClass distribution after Tomek:")
print(y_train_tomek.value_counts().sort_index())

model_tomek = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_tomek.fit(X_train_tomek, y_train_tomek)
y_pred_tomek = model_tomek.predict(X_test)

print("\n", classification_report(y_test, y_pred_tomek, digits=3))

# Explicit metrics for direct comparison against literature figures,
# which typically report accuracy or weighted F1, not macro-F1
acc = accuracy_score(y_test, y_pred_tomek)
weighted_f1 = f1_score(y_test, y_pred_tomek, average='weighted')
macro_f1 = f1_score(y_test, y_pred_tomek, average='macro')

print(f"\nAccuracy:     {acc:.4f}")
print(f"Weighted F1:  {weighted_f1:.4f}")
print(f"Macro F1:     {macro_f1:.4f}")

X_reduced shape: (160983, 59)
Before Tomek: 128786
After Tomek: 115873

Class distribution after Tomek:
attack_cat
0    64476
1     6501
2     1241
3     3251
4    19314
5     1282
6    13083
7      137
8      819
9     5769
Name: count, dtype: int64

               precision    recall  f1-score   support

           0      0.854     0.910     0.881     17098
           1      0.513     0.603     0.555      1949
           2      0.039     0.030     0.034       335
           3      0.219     0.196     0.207      1048
           4      0.759     0.790     0.774      5434
           5      0.279     0.048     0.082       354
           6      0.529     0.421     0.469      4139
           7      0.409     0.265     0.321        34
           8      0.518     0.349     0.417       292
           9      0.975     0.823     0.893      1514

    accuracy                          0.756     32197
   macro avg      0.509     0.443     0.463     32197
weighted avg      0.742     0.756     0.746

Baseline (k=5, euclidean, uniform):                  macro-F1 = 0.418
+ distance weighting:                                 macro-F1 = 0.423
+ SMOTE (flat target=5000):                            macro-F1 = 0.405  (worse — abandoned)
k/metric sweep -> Manhattan, k=7, weighted:            macro-F1 = 0.442
+ connection-cluster reduced (7->2 features):          macro-F1 = 0.446
+ trans_depth clamped/rescaled:                        macro-F1 = 0.447  (negligible — reverted, left unclamped)
+ Tomek links standalone (fold 1 only):                macro-F1 = 0.463  (~flat vs. above, class 5 recall collapsed)

CURRENT BEST CONFIRMED: Manhattan, k=7, weighted, reduced cluster, trans_depth untouched
                        macro-F1 ≈ 0.446 (5-fold confirmed)Baseline (k=5, euclidean, uniform):                  macro-F1 = 0.418
+ distance weighting:                                 macro-F1 = 0.423
+ SMOTE (flat target=5000):                            macro-F1 = 0.405  (worse — abandoned)
k/metric sweep -> Manhattan, k=7, weighted:            macro-F1 = 0.442
+ connection-cluster reduced (7->2 features):          macro-F1 = 0.446
+ trans_depth clamped/rescaled:                        macro-F1 = 0.447  (negligible — reverted, left unclamped)
+ Tomek links standalone (fold 1 only):                macro-F1 = 0.463  (~flat vs. above, class 5 recall collapsed)

CURRENT BEST CONFIRMED: Manhattan, k=7, weighted, reduced cluster, trans_depth untouched
                        macro-F1 ≈ 0.446 (5-fold confirmed)

In [ ]:
# ============================================================
# Cell 13: Feature selection sweep (filter method)
# ============================================================
# Rank all 59 features by relevance to attack_cat using mutual
# information (Topic 2 slide 14: filter method — statistical
# test between descriptive feature and target, independent of
# the model). Then test performance keeping only the top N,
# for several values of N, to see if fewer, more relevant
# features actually improve distance-based classification.

from sklearn.feature_selection import SelectKBest, mutual_info_classif

train_idx, test_idx = next(skf.split(X_reduced, y))
X_train, X_test = X_reduced.iloc[train_idx], X_reduced.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Rank features once
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_ranking = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

print("Top 15 most relevant features:")
print(mi_ranking.head(15))
print("\nBottom 15 least relevant features:")
print(mi_ranking.tail(15))

# Test different feature counts
feature_counts = [10, 20, 30, 40, 59]  # 59 = all features, baseline

for n in feature_counts:
    top_features = mi_ranking.head(n).index.tolist()
    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train[top_features], y_train)
    y_pred = model.predict(X_test[top_features])
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    print(f"\ntop {n:3d} features: macro-F1 = {macro_f1:.4f}")

Top 15 most relevant features:
sbytes          0.793321
dbytes          0.581593
smean           0.548915
dmean           0.456730
sload           0.426142
sttl            0.419877
ct_state_ttl    0.393153
dttl            0.381606
dload           0.359884
dinpkt          0.354935
dpkts           0.348970
rate            0.345673
synack          0.325194
dur             0.318268
ackdat          0.300042
dtype: float64

Bottom 15 least relevant features:
service_dhcp       0.004025
is_sm_ips_ports    0.003619
state_REQ          0.002763
service_ssl        0.001795
service_ftp        0.001385
state_URN          0.001036
state_CLO          0.000723
state_ECO          0.000621
state_PAR          0.000468
state_ACC          0.000441
service_snmp       0.000125
proto_arp          0.000000
state_RST          0.000000
service_radius     0.000000
service_irc        0.000000
dtype: float64

top  10 features: macro-F1 = 0.5097

top  20 features: macro-F1 = 0.4847

top  30 features: macro-F1 = 0.47

In [ ]:
# ============================================================
# Cell 14: Confirm top-10 feature selection with proper 5-fold CV
# ============================================================
fold_f1_selected = []
fold_acc_selected = []
fold_weighted_f1_selected = []
selected_features_per_fold = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_reduced, y)):
    X_train, X_test = X_reduced.iloc[train_idx], X_reduced.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    mi_ranking = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)
    top_10 = mi_ranking.head(10).index.tolist()
    selected_features_per_fold.append(top_10)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train[top_10], y_train)
    y_pred = model.predict(X_test[top_10])

    macro_f1 = f1_score(y_test, y_pred, average='macro')
    acc = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')

    fold_f1_selected.append(macro_f1)
    fold_acc_selected.append(acc)
    fold_weighted_f1_selected.append(weighted_f1)

    print(f"Fold {fold_num+1}: macro-F1={macro_f1:.4f}, accuracy={acc:.4f}, weighted-F1={weighted_f1:.4f}")
    print(f"  top features: {top_10}")

print(f"\nMean macro-F1:    {np.mean(fold_f1_selected):.4f} (± {np.std(fold_f1_selected):.4f})")
print(f"Mean accuracy:    {np.mean(fold_acc_selected):.4f} (± {np.std(fold_acc_selected):.4f})")
print(f"Mean weighted-F1: {np.mean(fold_weighted_f1_selected):.4f} (± {np.std(fold_weighted_f1_selected):.4f})")

Fold 1: macro-F1=0.5097, accuracy=0.7756, weighted-F1=0.7681
  top features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']
Fold 2: macro-F1=0.4975, accuracy=0.7751, weighted-F1=0.7681
  top features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']
Fold 3: macro-F1=0.4901, accuracy=0.7741, weighted-F1=0.7729
  top features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']
Fold 4: macro-F1=0.5078, accuracy=0.7760, weighted-F1=0.7692
  top features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']
Fold 5: macro-F1=0.5512, accuracy=0.7946, weighted-F1=0.7916
  top features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dpkts']

Mean macro-F1:    0.5113 (± 0.0212)
Mean accuracy:    0.7791 (± 0.0078)
Mean weighted-F1: 0.7740 (± 0.0090)


Baseline (k=5, euclidean, uniform):                    macro-F1 = 0.418
+ distance weighting:                                   macro-F1 = 0.423
+ SMOTE (flat target=5000):                              macro-F1 = 0.405  (worse — abandoned)
k/metric sweep -> Manhattan, k=7, weighted:              macro-F1 = 0.442
+ connection-cluster reduced (7->2 features):            macro-F1 = 0.446
+ trans_depth clamped/rescaled:                          macro-F1 = 0.447  (negligible — reverted)
+ Tomek links standalone:                                macro-F1 = 0.463  (~flat, class 5 hurt — set aside)
+ MI filter feature selection, top-10:                   macro-F1 = 0.511  <- NEW BEST

DECISION: reduce to 10 features (sbytes, dbytes, smean, dmean, sload,
sttl, ct_state_ttl, dttl, dload, dinpkt/dpkts), selected via mutual
information filter method (Topic 2, slide 15). Confirmed via proper
per-fold recomputation to avoid leakage. Feature set highly stable
across folds (4/5 identical). Real gain over full 59-feature set.

In [ ]:
# ============================================================
# Cell 15: Re-tune k + distance metric on the reduced 10-feature set
# ============================================================
top_10_stable = ['sbytes', 'dbytes', 'smean', 'dmean', 'sload',
                  'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']

train_idx, test_idx = next(skf.split(X_reduced, y))
X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

k_values = [1, 3, 5, 7, 9, 15, 21, 31]
metrics = ['euclidean', 'manhattan']

results = []
for metric in metrics:
    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=metric)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        results.append({'k': k, 'metric': metric, 'macro_f1': macro_f1})
        print(f"metric={metric:10s} k={k:3d}  macro-F1={macro_f1:.4f}")

results_df = pd.DataFrame(results)
best = results_df.loc[results_df['macro_f1'].idxmax()]
print(f"\nBest: metric={best['metric']}, k={best['k']}, macro-F1={best['macro_f1']:.4f}")

metric=euclidean  k=  1  macro-F1=0.4785
metric=euclidean  k=  3  macro-F1=0.4983
metric=euclidean  k=  5  macro-F1=0.5002
metric=euclidean  k=  7  macro-F1=0.5032
metric=euclidean  k=  9  macro-F1=0.4981
metric=euclidean  k= 15  macro-F1=0.4913
metric=euclidean  k= 21  macro-F1=0.4833
metric=euclidean  k= 31  macro-F1=0.4809
metric=manhattan  k=  1  macro-F1=0.4839
metric=manhattan  k=  3  macro-F1=0.5077
metric=manhattan  k=  5  macro-F1=0.5015
metric=manhattan  k=  7  macro-F1=0.5097
metric=manhattan  k=  9  macro-F1=0.5028
metric=manhattan  k= 15  macro-F1=0.5046
metric=manhattan  k= 21  macro-F1=0.4912
metric=manhattan  k= 31  macro-F1=0.4900

Best: metric=manhattan, k=7, macro-F1=0.5097


In [ ]:
# ============================================================
# Cell 16: Confirm k=7, Manhattan on the reduced 10-feature set
# with proper 5-fold CV
# ============================================================
top_10_stable = ['sbytes', 'dbytes', 'smean', 'dmean', 'sload',
                  'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']

fold_f1_final = []
fold_acc_final = []
fold_weighted_f1_final = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_reduced, y)):
    X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    macro_f1 = f1_score(y_test, y_pred, average='macro')
    acc = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')

    fold_f1_final.append(macro_f1)
    fold_acc_final.append(acc)
    fold_weighted_f1_final.append(weighted_f1)

    print(f"Fold {fold_num+1}: macro-F1={macro_f1:.4f}, accuracy={acc:.4f}, weighted-F1={weighted_f1:.4f}")

print(f"\nMean macro-F1:    {np.mean(fold_f1_final):.4f} (± {np.std(fold_f1_final):.4f})")
print(f"Mean accuracy:    {np.mean(fold_acc_final):.4f} (± {np.std(fold_acc_final):.4f})")
print(f"Mean weighted-F1: {np.mean(fold_weighted_f1_final):.4f} (± {np.std(fold_weighted_f1_final):.4f})")

Fold 1: macro-F1=0.5097, accuracy=0.7756, weighted-F1=0.7681
Fold 2: macro-F1=0.4975, accuracy=0.7751, weighted-F1=0.7681
Fold 3: macro-F1=0.4901, accuracy=0.7741, weighted-F1=0.7729
Fold 4: macro-F1=0.5078, accuracy=0.7760, weighted-F1=0.7692
Fold 5: macro-F1=0.5076, accuracy=0.7754, weighted-F1=0.7717

Mean macro-F1:    0.5025 (± 0.0075)
Mean accuracy:    0.7752 (± 0.0006)
Mean weighted-F1: 0.7700 (± 0.0020)


Baseline (k=5, euclidean, uniform):                    macro-F1 = 0.418
+ distance weighting:                                   macro-F1 = 0.423
k/metric sweep -> Manhattan, k=7, weighted:              macro-F1 = 0.442
+ connection-cluster reduced (7->2 features):            macro-F1 = 0.446
+ MI filter feature selection, top-10 (per-fold ranked): macro-F1 = 0.511 (±0.0212)
+ re-tuned k/metric on 10-feature set, confirmed:        macro-F1 = 0.5025 (±0.0075)  <- CURRENT BEST, stable

CONFIG: Manhattan distance, k=7, distance-weighted, 10 features
(sbytes, dbytes, smean, dmean, sload, sttl, ct_state_ttl, dttl,
dload, dinpkt), selected via mutual information filter method.

In [ ]:
# ============================================================
# Cell 17: Recalibrated SMOTE — ratio-capped, on 10-feature set
# ============================================================
# Previous attempt used a flat target (5000) for every class,
# which stretched class 7 by 36x and collapsed its precision.
# This time, cap the oversampling multiplier at 5x each class's
# original count instead, so no class gets stretched unrealistically
# thin. Tested on a single fold first, same pattern as before.

from imblearn.over_sampling import SMOTE
train_idx, test_idx = next(skf.split(X_reduced, y))
X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Before SMOTE:")
print(y_train.value_counts().sort_index())

# Cap each class's target at 5x its original count
MAX_RATIO = 5
target_counts = {cls: min(count * MAX_RATIO, y_train.value_counts().max())
                  for cls, count in y_train.value_counts().items()}

print("\nTarget counts (capped at 5x original):")
print(target_counts)

smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(y_train_smote.value_counts().sort_index())

model_smote = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = model_smote.predict(X_test)

macro_f1 = f1_score(y_test, y_pred_smote, average='macro')
print(f"\nmacro-F1 = {macro_f1:.4f} (vs. current best 0.5097 single-fold)")
print("\n", classification_report(y_test, y_pred_smote, digits=3))

Before SMOTE:
attack_cat
0    68389
1     7796
2     1343
3     4188
4    21736
5     1416
6    16559
7      137
8     1164
9     6058
Name: count, dtype: int64

Target counts (capped at 5x original):
{0: 68389, 4: 68389, 6: 68389, 1: 38980, 9: 30290, 3: 20940, 5: 7080, 2: 6715, 8: 5820, 7: 685}

After SMOTE:
attack_cat
0    68389
1    38980
2     6715
3    20940
4    68389
5     7080
6    68389
7      685
8     5820
9    30290
Name: count, dtype: int64

macro-F1 = 0.5118 (vs. current best 0.5097 single-fold)

               precision    recall  f1-score   support

           0      0.916     0.843     0.878     17098
           1      0.662     0.741     0.699      1949
           2      0.200     0.113     0.145       335
           3      0.305     0.403     0.347      1048
           4      0.727     0.761     0.743      5434
           5      0.164     0.133     0.147       354
           6      0.511     0.605     0.554      4139
           7      0.415     0.500     0.453       

In [ ]:
# ============================================================
# Cell 18: Confirm SMOTE (ratio-capped) with proper 5-fold CV
# ============================================================
fold_f1_smote_final = []
fold_acc_smote_final = []
fold_weighted_f1_smote_final = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_reduced, y)):
    X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    target_counts = {cls: min(count * 5, y_train.value_counts().max())
                      for cls, count in y_train.value_counts().items()}
    smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)

    macro_f1 = f1_score(y_test, y_pred, average='macro')
    acc = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')

    fold_f1_smote_final.append(macro_f1)
    fold_acc_smote_final.append(acc)
    fold_weighted_f1_smote_final.append(weighted_f1)
    print(f"Fold {fold_num+1}: macro-F1={macro_f1:.4f}, accuracy={acc:.4f}, weighted-F1={weighted_f1:.4f}")

print(f"\nMean macro-F1:    {np.mean(fold_f1_smote_final):.4f} (± {np.std(fold_f1_smote_final):.4f})")
print(f"Mean accuracy:    {np.mean(fold_acc_smote_final):.4f} (± {np.std(fold_acc_smote_final):.4f})")
print(f"Mean weighted-F1: {np.mean(fold_weighted_f1_smote_final):.4f} (± {np.std(fold_weighted_f1_smote_final):.4f})")

Fold 1: macro-F1=0.5118, accuracy=0.7586, weighted-F1=0.7645
Fold 2: macro-F1=0.5008, accuracy=0.7561, weighted-F1=0.7633
Fold 3: macro-F1=0.5181, accuracy=0.7533, weighted-F1=0.7650
Fold 4: macro-F1=0.5079, accuracy=0.7573, weighted-F1=0.7645
Fold 5: macro-F1=0.5110, accuracy=0.7565, weighted-F1=0.7670

Mean macro-F1:    0.5099 (± 0.0056)
Mean accuracy:    0.7564 (± 0.0018)
Mean weighted-F1: 0.7649 (± 0.0012)


FINAL CONFIG: Manhattan distance, k=7, distance-weighted,
10 features (MI filter-selected: sbytes, dbytes, smean, dmean,
sload, sttl, ct_state_ttl, dttl, dload, dinpkt),
SMOTE oversampling capped at 5x per class.

Mean macro-F1:    0.5099 (± 0.0056)
Mean accuracy:    0.7564 (± 0.0018)
Mean weighted-F1: 0.7649 (± 0.0012)

In [ ]:
# ============================================================
# Cell 19: Plain random over-sampling (Topic 2, slides 130-132)
# ============================================================
# Different mechanism than SMOTE — this duplicates existing
# minority rows (with replacement) rather than generating new
# synthetic points via interpolation. Sidesteps SMOTE's
# "bad synthetic point" risk entirely, at the cost of creating
# literal duplicate rows within the training set (not across
# train/test, so no leakage — this happens after the split).

from imblearn.over_sampling import RandomOverSampler

train_idx, test_idx = next(skf.split(X_reduced, y))
X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Before:")
print(y_train.value_counts().sort_index())

# Same 5x ratio cap as the SMOTE test, for a fair comparison
target_counts = {cls: min(count * 5, y_train.value_counts().max())
                  for cls, count in y_train.value_counts().items()}

ros = RandomOverSampler(sampling_strategy=target_counts, random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

print("\nAfter:")
print(y_train_ros.value_counts().sort_index())

model_ros = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_ros.fit(X_train_ros, y_train_ros)
y_pred_ros = model_ros.predict(X_test)

macro_f1 = f1_score(y_test, y_pred_ros, average='macro')
print(f"\nmacro-F1 = {macro_f1:.4f} (vs. SMOTE-capped 0.5118 single-fold)")
print("\n", classification_report(y_test, y_pred_ros, digits=3))

Before:
attack_cat
0    68389
1     7796
2     1343
3     4188
4    21736
5     1416
6    16559
7      137
8     1164
9     6058
Name: count, dtype: int64

After:
attack_cat
0    68389
1    38980
2     6715
3    20940
4    68389
5     7080
6    68389
7      685
8     5820
9    30290
Name: count, dtype: int64

macro-F1 = 0.5112 (vs. SMOTE-capped 0.5118 single-fold)

               precision    recall  f1-score   support

           0      0.923     0.837     0.878     17098
           1      0.694     0.731     0.712      1949
           2      0.192     0.119     0.147       335
           3      0.256     0.467     0.331      1048
           4      0.745     0.745     0.745      5434
           5      0.158     0.088     0.113       354
           6      0.501     0.612     0.551      4139
           7      0.463     0.559     0.507        34
           8      0.231     0.260     0.245       292
           9      0.901     0.867     0.884      1514

    accuracy                       

In [ ]:
# ============================================================
# Cell 20: Confirm plain random oversampling with proper 5-fold CV
# ============================================================
fold_f1_ros = []
fold_acc_ros = []
fold_weighted_f1_ros = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_reduced, y)):
    X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    target_counts = {cls: min(count * 5, y_train.value_counts().max())
                      for cls, count in y_train.value_counts().items()}
    ros = RandomOverSampler(sampling_strategy=target_counts, random_state=42)
    X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_ros, y_train_ros)
    y_pred = model.predict(X_test)

    macro_f1 = f1_score(y_test, y_pred, average='macro')
    acc = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')

    fold_f1_ros.append(macro_f1)
    fold_acc_ros.append(acc)
    fold_weighted_f1_ros.append(weighted_f1)
    print(f"Fold {fold_num+1}: macro-F1={macro_f1:.4f}, accuracy={acc:.4f}, weighted-F1={weighted_f1:.4f}")

print(f"\nMean macro-F1:    {np.mean(fold_f1_ros):.4f} (± {np.std(fold_f1_ros):.4f})")
print(f"Mean accuracy:    {np.mean(fold_acc_ros):.4f} (± {np.std(fold_acc_ros):.4f})")
print(f"Mean weighted-F1: {np.mean(fold_weighted_f1_ros):.4f} (± {np.std(fold_weighted_f1_ros):.4f})")

Fold 1: macro-F1=0.5112, accuracy=0.7543, weighted-F1=0.7636
Fold 2: macro-F1=0.4969, accuracy=0.7544, weighted-F1=0.7654
Fold 3: macro-F1=0.5050, accuracy=0.7505, weighted-F1=0.7638
Fold 4: macro-F1=0.5093, accuracy=0.7542, weighted-F1=0.7629
Fold 5: macro-F1=0.5051, accuracy=0.7532, weighted-F1=0.7642

Mean macro-F1:    0.5055 (± 0.0049)
Mean accuracy:    0.7533 (± 0.0015)
Mean weighted-F1: 0.7640 (± 0.0008)


Baseline (k=5, euclidean, uniform):                    macro-F1 = 0.418
+ distance weighting:                                   macro-F1 = 0.423
k/metric sweep -> Manhattan, k=7, weighted:              macro-F1 = 0.442
+ connection-cluster reduced (7->2 features):            macro-F1 = 0.446
+ MI filter feature selection, top-10:                   macro-F1 = 0.511 / 0.503 (fixed list)
+ SMOTE (ratio-capped at 5x):                             macro-F1 = 0.5099 (±0.0056)  <- current best
+ Plain random oversampling (ratio-capped at 5x):         macro-F1 = 0.5055 (±0.0049)  (comparable, not adopted)

In [ ]:
# ============================================================
# Cell 21: SMOTE + Tomek combined
# ============================================================
from imblearn.combine import SMOTETomek

train_idx, test_idx = next(skf.split(X_reduced, y))
X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

target_counts = {cls: min(count * 5, y_train.value_counts().max())
                  for cls, count in y_train.value_counts().items()}
smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)

smote_tomek = SMOTETomek(smote=smote, random_state=42)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train, y_train)

print("After SMOTE+Tomek:")
print(y_train_st.value_counts().sort_index())

model_st = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_st.fit(X_train_st, y_train_st)
y_pred_st = model_st.predict(X_test)

macro_f1 = f1_score(y_test, y_pred_st, average='macro')
print(f"\nmacro-F1 = {macro_f1:.4f} (vs. SMOTE alone 0.5118 single-fold)")
print("\n", classification_report(y_test, y_pred_st, digits=3))

After SMOTE+Tomek:
attack_cat
0    64836
1    37820
2     6247
3    19532
4    66025
5     6703
6    64531
7      609
8     4934
9    29904
Name: count, dtype: int64

macro-F1 = 0.5115 (vs. SMOTE alone 0.5118 single-fold)

               precision    recall  f1-score   support

           0      0.921     0.837     0.877     17098
           1      0.677     0.739     0.707      1949
           2      0.201     0.152     0.173       335
           3      0.281     0.466     0.351      1048
           4      0.752     0.758     0.755      5434
           5      0.138     0.071     0.093       354
           6      0.510     0.619     0.559      4139
           7      0.419     0.529     0.468        34
           8      0.223     0.267     0.243       292
           9      0.903     0.873     0.888      1514

    accuracy                          0.759     32197
   macro avg      0.503     0.531     0.511     32197
weighted avg      0.781     0.759     0.767     32197



+ SMOTE (ratio-capped at 5x):                    macro-F1 = 0.5099 (±0.0056)  <- STILL BEST
+ Plain random oversampling (ratio-capped):       macro-F1 = 0.5055 (±0.0049)  (comparable)
+ SMOTE+Tomek combined:                            macro-F1 ≈ 0.511 single-fold (comparable, class 5 hurt again)

In [ ]:
# ============================================================
# Cell 22: Min-max (current) vs. z-score, single-fold screen
# ============================================================
# z-score applied on top of already min-max-scaled columns is
# mathematically equivalent to z-scoring the original raw values
# directly (both are linear transforms). Scaler fit ONLY on the
# training fold, to avoid leaking test-fold statistics.

from sklearn.preprocessing import StandardScaler

train_idx, test_idx = next(skf.split(X_reduced, y))
X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Current best (min-max, already applied) — for reference
model_minmax = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_minmax.fit(X_train, y_train)
y_pred_minmax = model_minmax.predict(X_test)
macro_f1_minmax = f1_score(y_test, y_pred_minmax, average='macro')
print(f"Min-max (current):  macro-F1 = {macro_f1_minmax:.4f}")

# Z-score, fit on training fold only
scaler = StandardScaler()
X_train_z = scaler.fit_transform(X_train)
X_test_z = scaler.transform(X_test)

model_z = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_z.fit(X_train_z, y_train)
y_pred_z = model_z.predict(X_test_z)
macro_f1_z = f1_score(y_test, y_pred_z, average='macro')
print(f"Z-score:             macro-F1 = {macro_f1_z:.4f}")

print(f"\nDifference: {macro_f1_z - macro_f1_minmax:+.4f}")

Min-max (current):  macro-F1 = 0.5097
Z-score:             macro-F1 = 0.5031

Difference: -0.0066


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    classification_report,
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

In [ ]:
# ============================================================
# Quick setup: recreate everything this cell needs, no fitting
# ============================================================
cluster_cols = ['ct_srv_src', 'ct_srv_dst', 'ct_dst_src_ltm', 'ct_src_dport_ltm',
                 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm']
reduced_cluster_cols = ['ct_srv_src', 'ct_srv_dst']
drop_cols = [c for c in cluster_cols if c not in reduced_cluster_cols]

X_reduced = X.drop(columns=drop_cols)

top_10_stable = ['sbytes', 'dbytes', 'smean', 'dmean', 'sload',
                  'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']

print("X_reduced shape:", X_reduced.shape)



X_reduced shape: (160983, 59)


In [ ]:
# ============================================================
# Cell 23: Final per-class report — exact locked-in configuration,
# for the Results section
# ============================================================
all_y_test = []
all_y_pred = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_reduced, y)):
    X_train, X_test = X_reduced.iloc[train_idx][top_10_stable], X_reduced.iloc[test_idx][top_10_stable]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    target_counts = {cls: min(count * 5, y_train.value_counts().max())
                      for cls, count in y_train.value_counts().items()}
    smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)

    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)

print(classification_report(all_y_test, all_y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.918     0.838     0.876     85487
           1      0.678     0.743     0.709      9745
           2      0.218     0.145     0.174      1678
           3      0.268     0.443     0.334      5236
           4      0.749     0.748     0.749     27170
           5      0.143     0.098     0.116      1770
           6      0.508     0.613     0.555     20698
           7      0.358     0.509     0.420       171
           8      0.254     0.291     0.271      1456
           9      0.916     0.869     0.892      7572

    accuracy                          0.756    160983
   macro avg      0.501     0.530     0.510    160983
weighted avg      0.779     0.756     0.765    160983

